# SIF 아카이브 신버전 변환 노트북

원본 CSV/XLSX → 열구조 완성 → 엑셀 추출 → JSON 변환 순서로 진행합니다.

각 단계 끝에 `assert` 검증이 붙어 있습니다. 셀을 위에서부터 순서대로 실행하고, 오류 없이 끝나면 그 단계는 통과입니다.

**원칙: 원문에서 추출되지 않는 값은 채우지 않고 비워 둡니다.**

| 단계 | 내용 | 산출 |
|---|---|---|
| 0 | 준비·경로 설정 | |
| 1 | 신버전 원본 로드·정규화 | `new` 3,459행 |
| 2 | 구버전에서 룩업 규칙 역산 | 룩업 테이블 |
| 3 | 기인물 크로스워크·승계 컬럼 | 4개 열 |
| 4 | 발생시점 파생 | 4개 열 |
| 5 | KOEN공정 | 1개 열 |
| 6 | 분류코드 | 3개 열 |
| 7 | 추락고 추출 | 1개 열 |
| 8 | 재해정도·텍스트 파생 | 5개 열 |
| 9 | 열구조 완성 | 15열/28열 2종 |
| 10 | 엑셀 저장 | xlsx 2개 |
| 11 | JSON 변환 | json 2개 |
| 12 | 최종 검증 | |

## 0. 준비

`SRC` 폴더에 원본 3개 파일을 넣고 경로만 맞추면 됩니다.

In [ ]:
import pandas as pd, numpy as np, re, json, os
from pathlib import Path

SRC = Path('.')          # 원본 파일이 있는 폴더
OUT = Path('./out')      # 결과 저장 폴더
OUT.mkdir(exist_ok=True)

F_NEW_XLSX = SRC / '한국산업안전보건공단_산업재해 고위험요인SIF 아카이브_20260401.xlsx'
F_OLD_JSON = SRC / '구_db.json'

print('pandas', pd.__version__)
for f in (F_NEW_XLSX, F_OLD_JSON):
    print(('있음' if f.exists() else '없음'), f.name)

## 1. 신버전 원본 로드·정규화

공단 원본은 1~4행이 제목·병합 헤더입니다. 5행부터가 데이터이고 A열은 비어 있습니다.
따라서 `header=None`으로 읽은 뒤 5행·B열부터 잘라냅니다.

In [ ]:
raw = pd.read_excel(F_NEW_XLSX, sheet_name=0, header=None)
print('원본 크기:', raw.shape)
display(raw.head(6))

new = raw.iloc[4:, 1:10].copy()
new.columns = ['연번','공종','작업명','단위작업명','재해종류','재해개요','기인물','재해유발요인','위험성감소대책']
new = new.dropna(subset=['재해개요']).reset_index(drop=True)
# 문자열 열의 앞뒤 공백 제거
# (pandas 3.x 는 문자열 열 dtype 이 object 가 아니므로 dtype 으로 거르면 안 됨)
for c in new.columns:
    if c != '연번':
        new[c] = new[c].astype(str).str.strip()

print('정규화 후:', new.shape)
assert len(new) == 3459, f'행수가 3,459가 아님: {len(new)}'
assert new.isna().sum().sum() == 0, '결측 존재'
print('통과 — 3,459행 9열, 결측 0')

## 2. 구버전에서 룩업 규칙 역산

구버전 JSON에 사람이 붙여 둔 5개 열의 규칙을 데이터로 확인합니다.
추측이 아니라 **구버전 2,574건에서 실제로 성립하는지** 검증한 뒤에만 사용합니다.

In [ ]:
old = pd.DataFrame(json.load(open(F_OLD_JSON, encoding='utf-8')))
print('구버전:', old.shape)
print(list(old.columns))

In [ ]:
# (1) 기인물 하나가 4개 열의 값을 결정하는지 확인 (빈칸도 하나의 값으로 세어 흔들림을 잡아냄)
COLS4 = ['기인물분류','12대기인물','위험도순위','3년간사고비중']
n_all = {c: old.groupby('기인물')[c].nunique(dropna=False) for c in COLS4}
for c in COLS4:
    n = n_all[c]
    print(f'{c:12} 값이 갈리는 기인물: {(n>1).sum()}건  {list(n[n>1].index)}')

# 갈리는 항목의 실제 분포를 눈으로 확인 — 구버전 원본에 남아 있는 입력 누락입니다
흔들림 = sorted({k for c in COLS4 for k in n_all[c][n_all[c]>1].index})
for k in 흔들림:
    print()
    print(f'[{k}]')
    print(old.loc[old['기인물']==k].groupby(COLS4, dropna=False).size().to_string())

In [ ]:
# (2) 최빈값으로 룩업 테이블 확정 (위에서 확인한 소수 1건씩을 다수값에 맞춤)
def mode1(s):
    m = s.mode(dropna=True)
    return m.iat[0] if len(m) else None

LOOKUP = old.groupby('기인물')[COLS4].agg(mode1)
LOOKUP['12대기인물'] = LOOKUP['12대기인물'].astype(bool)

print('룩업 테이블', LOOKUP.shape)
assert len(LOOKUP) == old['기인물'].nunique()
display(LOOKUP.head())

In [ ]:
# (2) 혹서기 규칙 검증 — 재해개요의 발생월로 결정되는지 교차표로 확인
MP = re.compile(r'(?:19|20)\d{2}\s*[년월.\-/]?\s*(\d{1,2})\s*[월년.\-/경]')
m_old = old['재해개요'].map(lambda s: (lambda m: int(m.group(1)) if m else None)(MP.search(str(s))))
display(pd.crosstab(m_old, old['혹서기']))
# 6~8월=혹서기, 12~2월=혹한기, 그 외='-' 가 예외 없이 성립해야 함
def rule(m):
    if pd.isna(m): return None
    m = int(m)
    return '혹서기' if m in (6,7,8) else '혹한기' if m in (12,1,2) else '-'
chk = m_old.map(rule)
ok = old.loc[chk.notna(), '혹서기'].eq(chk[chk.notna()]).all()
assert ok, '혹서기 규칙 불일치'
print('통과 — 혹서기 = 6~8월, 혹한기 = 12~2월, 그 외 -')

In [ ]:
# (3) KOEN공정 규칙 검증 — 제외 공종 확인
print(old.loc[old['KOEN공정']==False, '공종'].value_counts().to_string())
print()
print(old.loc[old['KOEN공정']==False, '단위작업명'].str.contains('양수발전용 댐').sum(), '건이 양수발전용 댐')
EX_GJ = ('8. 교량공사','9. 터널공사','10. 하천 및 항만공사')
pred = ~(old['공종'].isin(EX_GJ) | old['단위작업명'].str.contains('양수발전용 댐', na=False))
assert pred.equals(old['KOEN공정']), 'KOEN공정 규칙 불일치'
print('통과 — 교량·터널·하천항만 공종과 양수발전용 댐이 제외 대상')

## 3. 기인물 크로스워크·승계 컬럼

신버전에서 기인물 명칭이 바뀌었습니다. 명칭만 바뀐 14종은 연결하고,
판단이 필요한 3종은 **연결하지 않고 미배정으로 둡니다.**

In [ ]:
XW = {
 '고소작업대(차)'                    : '고소작업대',
 '지붕 채광판 등(선라이트, 슬레이트)' : '지붕(채광판,선라이트)',
 '철골구조물'                        : '철골(철골구조물)',
 '철골자재'                          : '철골(철골자재)',
 '굴착기(백호우)'                    : '굴착기',
 '트럭류'                            : '트럭',
 '기타 개구부'                       : '개구부(기타개구부)',
 '바닥개구부(자재인양구 등)'          : '개구부(바닥개구부)',
 '기타 구조물'                       : '기타구조물',
 '기타 가설구조물'                   : '기타가설구조물',
 '기타 건설장비'                     : '기타건설장비',
 '기타 건설기계기구'                 : '기타건설기계기구',
 '전주, 철탑'                        : '전주',
 '기타 자재(내부마감재,시멘트,마대)' : '기타자재(내부마감재,시멘트,마대)',
}
# 신버전 신규 기인물 3종 — 사용자 결정(26.9.8)
#   작업발판 일체형 거푸집 → 거푸집 룩업 승계 (12대, 순위 11)
#   타워크레인             → 신설, 12대 아님, 분류는 크레인류와 같은 '건설기계'
#   줄걸이 기구            → 신설, 12대 아님, 분류도 '줄걸이 기구'로 신설
XW['작업발판 일체형 거푸집'] = '거푸집'
NEW_ITEMS = {'타워크레인': '건설기계', '줄걸이 기구': '줄걸이 기구'}
PENDING = set()

new['_구명칭'] = new['기인물'].map(lambda x: None if x in NEW_ITEMS else XW.get(x, x))

미연결 = sorted(set(new.loc[new['_구명칭'].notna(), '_구명칭']) - set(LOOKUP.index))
print('룩업에 없는 구명칭:', 미연결)
assert not 미연결, '크로스워크 누락'
print('미배정 대상:', int(new['기인물'].isin(PENDING).sum()), '건')

In [ ]:
for col in ['기인물분류','12대기인물','위험도순위','3년간사고비중']:
    new[col] = new['_구명칭'].map(LOOKUP[col])

for k, cat in NEW_ITEMS.items():
    m = new['기인물'] == k
    new.loc[m, '기인물분류'] = cat
    new.loc[m, '12대기인물'] = False
    new.loc[m, '3년간사고비중'] = '해당없음'
new['기인물분류']    = new['기인물분류'].fillna('분류불능')
new['3년간사고비중'] = new['3년간사고비중'].fillna('해당없음')
new['12대기인물']    = new['12대기인물'].fillna(False).astype(bool)

print('12대기인물:', new['12대기인물'].value_counts(dropna=False).to_dict())
print('위험도순위 부여:', int(new['위험도순위'].notna().sum()))
assert new['12대기인물'].isna().sum() == 0 and (new['기인물분류'] == '미배정').sum() == 0
print('통과 — 미배정 0건')

## 4. 발생시점 파생

재해개요 원문의 연·월 표기에서 뽑습니다. 표기 오류(`2018년 09년경` 등)까지 잡도록 정규식을 넓게 씁니다.

In [ ]:
YP = re.compile(r'((?:19|20)\d{2})')
MP = re.compile(r'(?:19|20)\d{2}\s*[년월.\-/]?\s*(\d{1,2})\s*[월년.\-/경]')
g  = lambda P: (lambda s: (lambda m: int(m.group(1)) if m else None)(P.search(str(s))))

new['발생연도'] = new['재해개요'].map(g(YP))
new['발생월']   = new['재해개요'].map(g(MP))

SEASON = {12:'겨울',1:'겨울',2:'겨울',3:'봄',4:'봄',5:'봄',
          6:'여름',7:'여름',8:'여름',9:'가을',10:'가을',11:'가을'}
new['계절']   = new['발생월'].map(lambda m: None if pd.isna(m) else SEASON[int(m)])
new['혹서기'] = new['발생월'].map(lambda m: None if pd.isna(m) else
                 ('혹서기' if int(m) in (6,7,8) else '혹한기' if int(m) in (12,1,2) else '-'))

print('연도 결측', int(new['발생연도'].isna().sum()), '| 월 결측', int(new['발생월'].isna().sum()))
print(new['발생연도'].value_counts().sort_index().to_string())
assert new['발생월'].between(1,12).all(), '월 범위 오류'
assert new['발생연도'].isna().sum() == 0 and new['발생월'].isna().sum() == 0
print('통과 — 결측 0')

## 5. KOEN공정

2단계에서 검증한 구버전 규칙은 교량·터널·하천항만 제외였습니다. **터널공사는 KOEN공정에 포함**하기로 하셨으므로(26.9.8) 신버전에는 교량·하천항만·양수발전용 댐만 제외합니다.

In [ ]:
# 구버전 규칙은 교량·터널·하천항만 제외였으나, 사용자 결정(26.9.8)으로 터널공사는 KOEN공정에 포함한다.
EX_GJ_KOEN = ('8. 교량공사', '10. 하천 및 항만공사')
new['KOEN공정'] = ~(new['공종'].isin(EX_GJ_KOEN) | new['단위작업명'].str.contains('양수발전용 댐', na=False))
print('터널공사 KOEN공정:', new.loc[new['공종']=='9. 터널공사','KOEN공정'].unique())
print(new['KOEN공정'].value_counts().to_dict())
print(new.loc[~new['KOEN공정'], '공종'].value_counts().to_string())

## 6. 분류코드

공종·작업명·단위작업명 앞에 이미 붙어 있는 번호를 떼어냅니다. 새로 만드는 값이 아닙니다.

In [ ]:
new['공종코드']     = new['공종'].str.extract(r'^(\d+)\.')[0].astype(int)
new['작업명코드']   = new['작업명'].str.extract(r'^(\d+\.\d+)')[0]
new['단위작업코드'] = new['단위작업명'].str.extract(r'^(\d+\.\d+\.\d+)')[0]

for c in ['공종코드','작업명코드','단위작업코드']:
    print(c, '결측', int(new[c].isna().sum()))
    assert new[c].isna().sum() == 0
display(new[['공종','공종코드','작업명','작업명코드','단위작업명','단위작업코드']].head(3))

## 7. 추락고 추출

**높이를 지시하는 표기가 붙은 경우만** 뽑습니다.
앵커 없는 `8m` 같은 표기는 `L=8m`(부재 길이), `1.5m 간격` 등과 구분할 수 없으므로 제외하고 공란으로 둡니다.

| 인정 | 예 |
|---|---|
| `H≒` `H=` `h=` | `추락(H≒18.0m)` |
| `높이` `높이:` | `높이 약 27m` |
| `N m 아래` | `약 4.7m 아래 바닥으로` |

단위 뒤 `(?![mM])`는 `H≒100mm`(철근 높이 100밀리)를 100미터로 잘못 읽는 것을 막습니다.

In [ ]:
U  = r'(?:m|M|미터)(?![mM])'
HP = [re.compile(r'[Hh]\s*[≒=≈:：]\s*약?\s*([0-9]+(?:\.[0-9]+)?)\s*' + U),
      re.compile(r'높이\s*[:：]?\s*약?\s*([0-9]+(?:\.[0-9]+)?)\s*' + U),
      re.compile(r'약?\s*([0-9]+(?:\.[0-9]+)?)\s*' + U + r'\s*아래')]

def fall_height(s):
    for P in HP:
        m = P.search(str(s))
        if m: return float(m.group(1))
    return None                      # 앵커 없으면 비움

new['추락고_m'] = new['재해개요'].map(fall_height)

print('추출', int(new['추락고_m'].notna().sum()), '건 / 공란', int(new['추락고_m'].isna().sum()))
r = new[new['재해종류']=='추락']
print(f"추락 사례 {r['추락고_m'].notna().sum()}/{len(r)} ({r['추락고_m'].notna().mean()*100:.1f}%)")
display(new['추락고_m'].describe().round(2))

In [ ]:
# 눈으로 검증 — 50m 초과 값이 실제 원문과 맞는지 직접 확인
for _, x in new[new['추락고_m'] > 50].head(10).iterrows():
    print(f"{x['추락고_m']:>7}m | ...{x['재해개요'][-55:]}")

## 8. 재해정도·텍스트 파생

재해정도는 원문 표현만 봅니다. 표현이 없으면 사망으로 채우지 않고 **미상**으로 둡니다.

In [ ]:
def degree(s):
    s = str(s)
    if '사망' in s or '익사' in s: return '사망'
    if '부상' in s:               return '부상'
    return '미상'

new['재해정도']        = new['재해개요'].map(degree)
new['복수재해자']      = new['재해개요'].str.contains(r'[2-9]\s*명|\d\d\s*명', regex=True)
new['감소대책_항목수'] = new['위험성감소대책'].str.count('▶')
new['재해개요_글자수'] = new['재해개요'].str.len()
new['익명처리']        = new['재해개요'].str.contains('○')

print(new['재해정도'].value_counts().to_dict())
print('복수재해자', int(new['복수재해자'].sum()), '| 익명처리', int(new['익명처리'].sum()))
print()
print('— 미상 20건 원문 끝부분 (사망으로 채우지 않은 근거) —')
for s in new.loc[new['재해정도']=='미상', '재해개요']:
    print('  ...' + s[-42:])

## 9. 열구조 완성

두 벌을 만듭니다. 행수는 같아야 합니다.

- **A. 기존 열구조 15열** — 사이트 교체용. 구버전 `구_db.json`과 열 이름·순서·타입 동일
- **B. 확장열 28열** — 논문 분석용

In [ ]:
new.insert(0, 'id', range(1, len(new)+1))

COLS_A = ['id','공종','작업명','단위작업명','KOEN공정','기인물분류','기인물','12대기인물',
          '위험도순위','3년간사고비중','혹서기','재해형태','재해개요','재해유발요인','위험성감소대책']
COLS_B = ['id','공종','작업명','단위작업명','공종코드','작업명코드','단위작업코드','KOEN공정',
          '기인물분류','기인물','12대기인물','위험도순위','3년간사고비중',
          '발생연도','발생월','계절','혹서기','재해형태','재해종류','재해정도','복수재해자','추락고_m',
          '재해개요','재해유발요인','위험성감소대책','감소대책_항목수','재해개요_글자수','익명처리']

# 재해형태 = 구버전 재해형태 용어체계로 표기 통일
# 규칙 A. 구버전에 같은 표기가 있으면 그대로 둔다 (감전·끼임·익사·폭발·화재)
# 규칙 B. 표기가 다르면 구버전에 실재하는 용어로 치환한다
#         → 치환 대상은 전부 구버전 20종 안에 있는 값이며, 새 용어를 만들지 않는다
# 규칙 C. 판단 근거가 부족한 3종은 치환하지 않고 원문값을 남긴다 (전도·화상·파열)
TERM_FIX = {
    '추락'        : '떨어짐',
    '낙하'        : '맞음',
    '붕괴'        : '무너짐',
    '깔림'        : '깔림 뒤집힘',
    '부딪힘'      : '부딪힘 접촉',
    '베임'        : '절단 베임 찔림',
    '찔림'        : '절단 베임 찔림',
    '질식'        : '산소결핍 질식',
    '중독'        : '유해위험물질 노출 접촉',
    '유해물 접촉' : '유해위험물질 노출 접촉',
    '이상온도 접촉': '이상온도 노출 접촉',
    '기타'        : '원인미상',
}
KEEP    = ['감전','끼임','익사','폭발','화재']          # 구버전에 같은 표기 존재
READ_TERM = ['전도','화상','파열']                      # 원문 판독으로 배정

구버전용어 = set(old['재해형태'].unique())
assert set(TERM_FIX.values()) <= 구버전용어, '구버전에 없는 용어를 만들어냄'
assert set(KEEP) <= 구버전용어

new['재해형태'] = new['재해종류'].replace(TERM_FIX)

print('구버전 용어 20종 :', sorted(구버전용어))
# ── 재해종류 라벨만으로는 구버전 용어를 정할 수 없는 3종은 재해개요 원문을 읽어 배정한다.
#    라벨을 통째로 옮기지 않고, 사고가 실제로 어떻게 끝났는지(결과절)에 나타난 기전을 따른다.
#    예: 전도 155건 중 사람이 떨어져 죽었으면 떨어짐, 장비에 깔려 죽었으면 깔림 뒤집힘.
#    기전이 서술되지 않은 건은 배정하지 않고 원문값을 남긴다.

기전어 = [('깔림 뒤집힘',   ['깔려','깔리','깔린','깔림','뒤집','협착','매몰','묻혀','끼여','끼어','끼인']),
          ('떨어짐',        ['떨어져','떨어지','추락','전락','실족']),
          ('맞음',          ['덮쳐','덮치','강타','가격','맞아','맞은','타격','치여']),
          ('부딪힘 접촉',   ['부딪','충돌','충격']),
          ('익사',          ['익사','물에 빠','수몰'])]

원인어 = [('감전',              ['아크','감전','충전부','지락','전차선']),
          ('폭발',              ['폭발','백드래프트']),
          ('이상온도 노출 접촉', ['스팀','수증기','용융','열수','고온']),
          ('화재',              ['화재','화염','불꽃','불티','불이 붙','옮겨 붙','점화'])]

def 결과절(s, 기준어):
    for a in 기준어:
        i = s.rfind(a)
        if i >= 0: return s[i:]
    return s[-70:]

def 마지막기전(tl, 사전):
    hits = [(tl.rfind(w), lab) for lab, ws in 사전 for w in ws if tl.rfind(w) >= 0]
    return max(hits)[1] if hits else None

def 첫원인(s, 사전):
    for lab, ws in 사전:
        if any(w in s for w in ws): return lab
    return None

배정, 검토 = {}, []
for _, x in new.iterrows():
    t = x['재해종류']
    if t == '전도':
        v = 마지막기전(결과절(x['재해개요'], ['전도','전복','넘어','쓰러','기울어']), 기전어)
    elif t == '파열':
        v = 마지막기전(결과절(x['재해개요'], ['파열','이탈','튀어']), 기전어)
    elif t == '화상':
        v = 첫원인(x['재해개요'], 원인어)
    else:
        continue
    배정[x.name] = v
    검토.append({'id': x['id'], '재해종류': t, '배정': v, '재해개요': x['재해개요']})

배정 = pd.Series(배정)
new.loc[배정.dropna().index, '재해형태'] = 배정.dropna().values

검토 = pd.DataFrame(검토)
print('원문 판독으로 배정한 3종')
print(검토.groupby(['재해종류','배정'], dropna=False).size().to_string())
print()
print('배정 못한 건(원문값 유지):', int(검토['배정'].isna().sum()))

# ── 결과절에 기전이 서술되지 않은 건은 재해개요 전문을 읽어 개별 판정한다.
#    판정 기준은 구버전 용어 정의를 따른다.
#      · 장비·구조물이 전도·전복되어 사람이 재해를 입음      -> 깔림 뒤집힘
#      · 사람이 스스로 균형을 잃고 같은 높이에서 넘어짐      -> 넘어짐
#      · 사람이 높은 곳에 있다가 지면까지 내려옴             -> 떨어짐
#      · 원문에 기전이 전혀 없음                            -> 원인미상
개별판정 = {
      25: ('깔림 뒤집힘', '덤프트럭이 성토사면 아래로 전도, 운전자 사망'),
      44: ('깔림 뒤집힘', '차량이 소단 아래로 전복'),
      46: ('깔림 뒤집힘', '덤프트럭과 함께 재해자가 전복'),
     186: ('깔림 뒤집힘', '적재함 상승 상태 덤프트럭이 옆으로 넘어짐'),
     402: ('넘어짐',      '피재자가 작업 중 뒤로 넘어짐'),
     519: ('깔림 뒤집힘', '벽체철근(높이 약 10m)이 전도'),
     575: ('넘어짐',      '콘크리트 바닥에서 넘어짐'),
     681: ('깔림 뒤집힘', '옥외 행사용 철구조물이 넘어짐'),
    1912: ('떨어짐',      '전주 승주 중 전주 파단, 전주와 함께 지상 바닥까지'),
    1918: ('떨어짐',      '전주 승주(h≒4.5m) 중 전주 파단, 재해자와 함께'),
    2441: ('넘어짐',      '피재자가 닻줄에 걸려 넘어짐'),
    2448: ('넘어짐',      '예인선 바닥에서 넘어짐'),
    3324: ('깔림 뒤집힘', '살수차가 슬로프를 밀려 내려와 전복'),
    3337: ('원인미상',    '베란다에 쓰러진 상태로 발견, 기전 서술 없음'),
    3371: ('깔림 뒤집힘', '벽에 세워둔 석재 대리석이 넘어짐'),
    3393: ('깔림 뒤집힘', '고소작업대가 전도'),
    3454: ('깔림 뒤집힘', '살수차가 경사로를 후진하여 저수지로 전복'),
}
구버전용어 = set(old['재해형태'].unique())
assert {v for v, _ in 개별판정.values()} <= 구버전용어, '구버전에 없는 용어를 만들어냄'

미판정 = 검토.loc[검토['배정'].isna(), 'id'].tolist()
assert set(미판정) == set(개별판정), f'개별판정 대상 불일치: {sorted(set(미판정) ^ set(개별판정))}'

for i, (v, _) in 개별판정.items():
    new.loc[new['id'] == i, '재해형태'] = v
    검토.loc[검토['id'] == i, '배정'] = v
검토['판정사유'] = 검토['id'].map({i: r for i, (_, r) in 개별판정.items()})

print('개별 판정 결과')
print(pd.Series([v for v, _ in 개별판정.values()]).value_counts().to_string())
print()
assert set(new['재해형태']) <= 구버전용어, '구버전 용어를 벗어난 값이 남아 있음'
assert new['재해형태'].isna().sum() == 0
print('재해형태 최종 — 전건이 구버전 20종 안에 있음, 결측 0')
검토.to_csv(OUT/'SIF_재해형태_원문판독_검토목록.csv', index=False, encoding='utf-8-sig')

print('검토목록 저장 — 원문·배정값·판정사유를 나란히 확인하실 수 있습니다')

print('신버전 재해종류별 처리')
for t in sorted(new['재해종류'].unique()):
    n = int((new['재해종류']==t).sum())
    v = new.loc[new['재해종류']==t, '재해형태'].iat[0]
    구분 = '유지' if t in KEEP else ('판독' if t in READ_TERM else '치환')
    print(f'  {t:<12} {n:>5}건  {구분}  -> {v}')

print()
print('재해개요 무결성 확인 :', new['재해개요'].equals(
      pd.read_excel(F_NEW_XLSX, sheet_name=0, header=None).iloc[4:,6]
        .dropna().astype(str).str.strip().reset_index(drop=True)))



A = new[COLS_A].copy()
B = new[COLS_B].copy()

print('A', A.shape, '| B', B.shape)
assert len(A) == len(B) == 3459
assert list(A.columns) == list(json.load(open(F_OLD_JSON, encoding='utf-8'))[0].keys()), '구버전과 열구조 불일치'
print('통과 — 행수 일치, A의 열구조가 구버전과 동일')

## 10. 엑셀 추출

첫 행 고정과 자동필터를 걸어 저장합니다.

In [ ]:
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

WID = {'id':6,'공종':18,'작업명':22,'단위작업명':28,'공종코드':9,'작업명코드':11,'단위작업코드':12,
       'KOEN공정':10,'기인물분류':14,'기인물':28,'12대기인물':11,'위험도순위':10,'3년간사고비중':13,
       '발생연도':9,'발생월':8,'계절':7,'혹서기':9,'재해형태':12,'재해종류':10,'재해정도':9,
       '복수재해자':10,'추락고_m':10,'재해개요':70,'재해유발요인':50,'위험성감소대책':70,
       '감소대책_항목수':13,'재해개요_글자수':13,'익명처리':9}

def save_xlsx(df, path, sheet):
    with pd.ExcelWriter(path, engine='openpyxl') as w:
        df.to_excel(w, sheet_name=sheet, index=False)
        ws = w.sheets[sheet]
        hf, hfill = Font(name='Arial', bold=True, color='FFFFFF', size=10), PatternFill('solid', fgColor='1F4E79')
        thin = Side(style='thin', color='D9D9D9'); bd = Border(left=thin, right=thin, top=thin, bottom=thin)
        for c in ws[1]:
            c.font, c.fill = hf, hfill
            c.alignment = Alignment(horizontal='center', vertical='center'); c.border = bd
        for row in ws.iter_rows(min_row=2):
            for c in row:
                c.font = Font(name='Arial', size=9); c.border = bd
        for i, col in enumerate(df.columns, 1):
            ws.column_dimensions[get_column_letter(i)].width = WID.get(col, 14)
        ws.freeze_panes = 'B2'
        ws.auto_filter.ref = f'A1:{get_column_letter(len(df.columns))}{len(df)+1}'
    print('저장', path)

save_xlsx(A, OUT/'SIF_신버전_기존열구조_3459건.xlsx', '기존열구조_15열')
save_xlsx(B, OUT/'SIF_신버전_확장열_3459건.xlsx',   '확장열_28열')

A.to_csv(OUT/'SIF_신버전_기존열구조_3459건.csv', index=False, encoding='utf-8-sig')
B.to_csv(OUT/'SIF_신버전_확장열_3459건.csv',   index=False, encoding='utf-8-sig')
print('CSV 저장 완료 (UTF-8 BOM)')

## 11. JSON 변환

pandas의 numpy 타입은 그대로 두면 `true` 대신 문자열이 되거나 정수가 `1.0`으로 나갑니다.
구버전 JSON과 타입을 맞추기 위해 파이썬 기본형으로 바꿉니다.

In [ ]:
INT_COLS = {'id','위험도순위','발생연도','발생월','감소대책_항목수','재해개요_글자수'}

def to_records(df):
    out = []
    for r in df.to_dict('records'):
        o = {}
        for k, v in r.items():
            if isinstance(v, (np.bool_, bool)):        v = bool(v)
            elif isinstance(v, np.integer):            v = int(v)
            elif isinstance(v, (float, np.floating)):  v = None if pd.isna(v) else (int(v) if k in INT_COLS else float(v))
            elif v is None or v is pd.NA:              v = None
            o[k] = v
        out.append(o)
    return out

recA, recB = to_records(A), to_records(B)
json.dump(recA, open(OUT/'SIF_신버전_기존열구조_3459건.json','w',encoding='utf-8'), ensure_ascii=False, indent=1)
json.dump(recB, open(OUT/'SIF_신버전_확장열_3459건.json','w',encoding='utf-8'),   ensure_ascii=False, indent=1)
print(len(recA), len(recB))
print(json.dumps(recA[0], ensure_ascii=False, indent=1))

## 12. 최종 검증

구버전 JSON과 타입을 하나씩 대조하고, 저장한 파일을 다시 읽어 원본과 같은지 확인합니다.

In [ ]:
old_rec = json.load(open(F_OLD_JSON, encoding='utf-8'))

print(f"{'컬럼':<14}{'구버전 타입':<24}{'신버전 타입'}")
for k in old_rec[0].keys():
    t_o = sorted({type(r[k]).__name__ for r in old_rec})
    t_n = sorted({type(r[k]).__name__ for r in recA})
    mark = '' if t_o == t_n else '   <-- 차이'
    print(f'{k:<14}{str(t_o):<24}{t_n}{mark}')

In [ ]:
# 저장한 파일 재읽기 → 원본과 값 일치 확인
chk_json  = pd.DataFrame(json.load(open(OUT/'SIF_신버전_기존열구조_3459건.json', encoding='utf-8')))
chk_xlsx  = pd.read_excel(OUT/'SIF_신버전_기존열구조_3459건.xlsx')

assert len(chk_json) == len(chk_xlsx) == 3459
assert list(chk_json.columns) == COLS_A
for c in ['공종','기인물','재해개요','혹서기']:
    assert chk_json[c].equals(chk_xlsx[c].astype(str)) or (chk_json[c] == chk_xlsx[c]).all(), c

print('행수      ', len(chk_json))
print('열수      ', len(chk_json.columns))
print('12대기인물 공란', int(chk_json['12대기인물'].isna().sum()), '(미배정 3종)')
print('추락고 공란   ', int(pd.DataFrame(recB)['추락고_m'].isna().sum()))
print()
print('전 단계 검증 통과')

## 남은 판단 사항

노트북이 자동으로 정할 수 없는 두 가지입니다.

**1. 기인물 3종 배정** — 3단계 `PENDING` 집합에서 빼고 `XW`에 대응 항목을 넣으면 채워집니다.

| 신버전 기인물 | 건수 | 검토 |
|---|---|---|
| 작업발판 일체형 거푸집 | 42 | 갱폼 개칭. 비계(일체형비계)인지 거푸집인지 |
| 타워크레인 | 31 | 구버전 고정식크레인과 같은 항목인지 |
| 줄걸이 기구 | 43 | 구버전에 대응 없음. 새 분류가 필요 |

**2. 재해형태 값 체계** — 9단계에서 `추락 → 떨어짐` 한 건만 치환했습니다. 나머지 재해종류 값은 원문 그대로입니다.
다른 용어까지 일괄 환원하려면 `TERM_FIX`에 항목을 추가하시면 됩니다.